# 14 - Submission Codabench

Notebook final para generar `prediction.json` y `submission.zip` a partir de un modelo ya entrenado. Por defecto reconstruye DenseNet121 transfer learning y carga los pesos del `13`, pero se puede adaptar a `11`, `12` o `10` cambiando la configuracion.


## Librerias y configuracion


In [1]:
import json
import time
import zipfile
from pathlib import Path

import numpy as np
import tensorflow as tf
from PIL import Image

RUN_START = time.perf_counter()
print('GPUs disponibles:', tf.config.list_physical_devices('GPU'))


2026-06-04 11:13:31.874189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780571612.118849      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780571612.185121      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780571612.703302      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780571612.703350      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780571612.703354      58 computation_placer.cc:177] computation placer alr

GPUs disponibles: []


2026-06-04 11:13:49.862032: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


### Configuracion del modelo


In [2]:
MODEL_KIND = 'densenet121_transfer'
MODEL_FILENAME = '13_densenet121_finetuned.weights.h5'
MODEL_PATH = None  # None busca el archivo en /kaggle/working y /kaggle/input
MODEL_LOAD_MODE = 'weights'  # 'weights' para .weights.h5, 'keras' para modelos .keras completos

OUTPUT_JSON_NAME = 'prediction_14.json'
OUTPUT_ZIP_NAME = 'submission_14.zip'
IMAGE_SHAPE = (224, 224, 3)


## Datos


### Lectura del JSON


In [3]:
KAGGLE_INPUT_DIR = Path('/kaggle/input')
WORKDIR = Path('/kaggle/working')

if not KAGGLE_INPUT_DIR.exists():
    raise FileNotFoundError('No existe /kaggle/input. Ejecuta este notebook en Kaggle.')

json_matches = sorted(KAGGLE_INPUT_DIR.rglob('xview_ann_train.json'))
if not json_matches:
    available_inputs = [p.name for p in KAGGLE_INPUT_DIR.iterdir() if p.is_dir()]
    raise FileNotFoundError(
        'No se encontro xview_ann_train.json. Anade el dataset original xview_recognition. '
        f'Inputs disponibles: {available_inputs}'
    )

ANN_JSON = json_matches[0]
DATA_ROOT = ANN_JSON.parent
TEST_DIR = DATA_ROOT / 'xview_test'

with open(ANN_JSON, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

categories = {int(k): v['name'] for k, v in sorted(json_data['categories'].items(), key=lambda item: int(item[0]))}
category_names = list(categories.values())
num_classes = len(category_names)

print('DATA_ROOT:', DATA_ROOT)
print('TEST_DIR exists:', TEST_DIR.exists())
print('num_classes:', num_classes)
print(category_names)

if MODEL_PATH is None:
    model_candidates = [WORKDIR / MODEL_FILENAME] + sorted(KAGGLE_INPUT_DIR.rglob(MODEL_FILENAME))
    model_candidates = [p for p in model_candidates if p.exists()]
    if not model_candidates:
        raise FileNotFoundError(
            f'No se encontro {MODEL_FILENAME}. Si ejecutas este notebook separado, anade el output del notebook 13 como Input.'
        )
    MODEL_PATH = model_candidates[0]
else:
    MODEL_PATH = Path(MODEL_PATH)

print('MODEL_PATH:', MODEL_PATH)


DATA_ROOT: /kaggle/input/datasets/eduardocama/xview-recognition1/xview_recognition
TEST_DIR exists: True
num_classes: 13
['Cargo plane', 'Small car', 'Bus', 'Truck', 'Motorboat', 'Fishing vessel', 'Dump truck', 'Excavator', 'Building', 'Helipad', 'Storage tank', 'Shipping container', 'Pylon']
MODEL_PATH: /kaggle/input/models/eduardocama/xview-densenet121-weights/keras/default/1/13_densenet121_finetuned.weights.h5


## Modelo


### Arquitecturas disponibles


In [4]:
def build_resnet50_transfer(num_classes):
    from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

    inputs = tf.keras.Input(shape=IMAGE_SHAPE)
    x = tf.keras.layers.Lambda(resnet_preprocess, name='resnet50_preprocess')(inputs)
    base = tf.keras.applications.ResNet50(include_top=False, weights=None, input_tensor=x)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='resnet50_transfer')


def build_efficientnetb0_transfer(num_classes):
    inputs = tf.keras.Input(shape=IMAGE_SHAPE)
    base = tf.keras.applications.EfficientNetB0(include_top=False, weights=None, input_tensor=inputs)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='efficientnetb0_transfer')


def build_densenet121_transfer(num_classes):
    from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

    inputs = tf.keras.Input(shape=IMAGE_SHAPE)
    x = tf.keras.layers.Lambda(densenet_preprocess, name='densenet121_preprocess')(inputs)
    base = tf.keras.applications.DenseNet121(include_top=False, weights=None, input_tensor=x)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(0.35)(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='densenet121_transfer')


def build_vgg16_scratch(num_classes):
    inputs = tf.keras.Input(shape=IMAGE_SHAPE)
    x = tf.keras.layers.Rescaling(1.0 / 255.0)(inputs)
    base = tf.keras.applications.VGG16(include_top=False, weights=None, input_tensor=x)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs, name='vgg16_scratch')


MODEL_BUILDERS = {
    'resnet50_transfer': build_resnet50_transfer,
    'efficientnetb0_transfer': build_efficientnetb0_transfer,
    'densenet121_transfer': build_densenet121_transfer,
    'vgg16_scratch': build_vgg16_scratch,
}


### Carga del modelo


In [5]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'No existe MODEL_PATH: {MODEL_PATH}')

if MODEL_LOAD_MODE == 'keras':
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
elif MODEL_LOAD_MODE == 'weights':
    if MODEL_KIND not in MODEL_BUILDERS:
        raise ValueError(f'MODEL_KIND no reconocido: {MODEL_KIND}. Opciones: {sorted(MODEL_BUILDERS)}')
    model = MODEL_BUILDERS[MODEL_KIND](num_classes)
    model.load_weights(MODEL_PATH)
else:
    raise ValueError("MODEL_LOAD_MODE debe ser 'weights' o 'keras'")

model.summary()
print('Modelo cargado:', MODEL_PATH)


Model: "densenet121_transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121_prepro… │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ densenet121_prep… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b

 Total params: 7,303,245 (27.86 MB)

 Trainable params: 7,219,597 (27.54 MB)

 Non-trainable params: 83,648 (326.75 KB)

Modelo cargado: /kaggle/input/models/eduardocama/xview-densenet121-weights/keras/default/1/13_densenet121_finetuned.weights.h5


## Codabench


### Generacion del submission


In [6]:
CODABENCH_START = time.perf_counter()

def load_test_image(filename):
    return np.array(Image.open(DATA_ROOT / filename).convert('RGB'))

test_files = sorted(TEST_DIR.rglob('*.tif'))
print('Number of testing images:', len(test_files))
assert len(test_files) > 0, 'No se encontraron imagenes .tif en TEST_DIR.'

predictions_data = {'images': {}, 'annotations': {}}
batch_size = 64

for start in range(0, len(test_files), batch_size):
    batch_paths = test_files[start:start + batch_size]
    json_filenames = [path.relative_to(TEST_DIR).as_posix() for path in batch_paths]
    load_filenames = [path.relative_to(DATA_ROOT).as_posix() for path in batch_paths]

    images = np.stack([load_test_image(filename) for filename in load_filenames]).astype(np.uint8)
    probs = model.predict(images, batch_size=batch_size, verbose=0)
    pred_idx = np.argmax(probs, axis=1)
    pred_scores = np.max(probs, axis=1)

    for offset, (path, json_filename, cls_idx, score) in enumerate(zip(batch_paths, json_filenames, pred_idx, pred_scores)):
        idx = start + offset
        predictions_data['images'][idx] = {
            'image_id': path.name,
            'filename': json_filename,
            'width': 224,
            'height': 224,
        }
        predictions_data['annotations'][idx] = {
            'image_id': path.name,
            'category_id': category_names[int(cls_idx)],
            'score': float(score),
            'bbox': [0, 0, 224, 224],
        }

    print(f'Procesadas {min(start + batch_size, len(test_files))}/{len(test_files)}')

assert len(predictions_data['images']) == len(test_files)
assert len(predictions_data['annotations']) == len(test_files)

prediction_path = WORKDIR / OUTPUT_JSON_NAME
submission_path = WORKDIR / OUTPUT_ZIP_NAME

with open(prediction_path, 'w', encoding='utf-8') as outfile:
    json.dump(predictions_data, outfile)

with zipfile.ZipFile(submission_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(prediction_path, arcname='prediction.json')

CODABENCH_TIME = time.perf_counter() - CODABENCH_START

print('Archivos generados:')
print(prediction_path, '-', prediction_path.stat().st_size, 'bytes')
print(submission_path, '-', submission_path.stat().st_size, 'bytes')
print(f'Tiempo Codabench: {CODABENCH_TIME:.2f} s')

from IPython.display import FileLink, display
display(FileLink(str(submission_path)))


Number of testing images: 2365
Procesadas 64/2365
Procesadas 128/2365
Procesadas 192/2365
Procesadas 256/2365
Procesadas 320/2365
Procesadas 384/2365
Procesadas 448/2365
Procesadas 512/2365
Procesadas 576/2365
Procesadas 640/2365
Procesadas 704/2365
Procesadas 768/2365
Procesadas 832/2365
Procesadas 896/2365
Procesadas 960/2365
Procesadas 1024/2365
Procesadas 1088/2365
Procesadas 1152/2365
Procesadas 1216/2365
Procesadas 1280/2365
Procesadas 1344/2365
Procesadas 1408/2365
Procesadas 1472/2365
Procesadas 1536/2365
Procesadas 1600/2365
Procesadas 1664/2365
Procesadas 1728/2365
Procesadas 1792/2365
Procesadas 1856/2365
Procesadas 1920/2365
Procesadas 1984/2365
Procesadas 2048/2365
Procesadas 2112/2365
Procesadas 2176/2365
Procesadas 2240/2365
Procesadas 2304/2365
Procesadas 2365/2365
Archivos generados:
/kaggle/working/prediction_14.json - 707228 bytes
/kaggle/working/submission_14.zip - 161241 bytes
Tiempo Codabench: 265.11 s


/kaggle/working/submission_14.zip

### Resumen de tiempos


In [7]:
TOTAL_TIME = time.perf_counter() - RUN_START
print('Resumen de tiempos')
if 'CODABENCH_TIME' in globals():
    print(f'- Codabench: {CODABENCH_TIME:.2f} s')
print(f'- Tiempo total Run All: {TOTAL_TIME:.2f} s')
print(f'- Tiempo total Run All: {TOTAL_TIME / 60:.2f} min')


Resumen de tiempos
- Codabench: 265.11 s
- Tiempo total Run All: 289.51 s
- Tiempo total Run All: 4.83 min
